# LECTURE 14 — HANDS-ON EXERCISE
### MODULE 6: NEURAL NETWORKS & DEEP LEARNING

**HANDS-ON EXERCISE** — Train a Neural Network for Banking Crisis Classification

*WAIFEM ML Facilitation Programme 2026 — Compatible with Google Colab & VS Code*

### Package Installation

Uncomment and run the cell below if any package is missing:

In [ ]:
# !pip install tensorflow scikit-learn matplotlib pandas numpy seaborn

## ── SECTION 1: IMPORTS

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay,
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 120})

try:
    import google.colab          # noqa: F401
    print("Environment : Google Colab")
except ImportError:
    print("Environment : Local (VS Code / terminal)")

print("Libraries loaded successfully.\n")

## ── SECTION 2: SYNTHETIC BANKING SECTOR DATA

In [ ]:
def generate_banking_data(n_obs: int = 600, seed: int = 42) -> pd.DataFrame:
    """
    Simulate 600 annual country-year observations.

    Features  : npl_ratio, capital_adequacy, liquidity_ratio, roa,
                credit_growth, inflation, gdp_growth, current_account
    Target    : banking_crisis  (1 = crisis, 0 = no crisis)
    Class mix : approx. 15 % crises  (imbalanced — as in real data)
    """
    rng = np.random.default_rng(seed)

    npl_ratio        = rng.gamma(shape=2.5, scale=2.5,  size=n_obs)          # %
    capital_adequacy = rng.normal(loc=13,   scale=3.0,  size=n_obs).clip(4, 30)   # %
    liquidity_ratio  = rng.normal(loc=30,   scale=8.0,  size=n_obs).clip(10, 70)  # %
    roa              = rng.normal(loc=1.5,  scale=1.2,  size=n_obs)          # %
    credit_growth    = rng.normal(loc=12,   scale=8.0,  size=n_obs)          # %
    inflation        = rng.gamma(shape=2,   scale=3.5,  size=n_obs)          # %
    gdp_growth       = rng.normal(loc=3.5,  scale=2.0,  size=n_obs)          # %
    current_account  = rng.normal(loc=-2,   scale=4.0,  size=n_obs)          # % of GDP

    # Crisis probability via logistic model
    logit = (
        -4.0
        + 0.25 * npl_ratio
        - 0.15 * capital_adequacy
        - 0.05 * liquidity_ratio
        - 0.40 * roa
        + 0.08 * credit_growth
        + 0.10 * inflation
        - 0.20 * gdp_growth
        - 0.08 * current_account
        + rng.normal(0, 0.5, n_obs)
    )
    prob   = 1.0 / (1.0 + np.exp(-logit))
    crisis = (rng.uniform(0, 1, n_obs) < prob).astype(int)

    return pd.DataFrame({
        'npl_ratio':        npl_ratio,
        'capital_adequacy': capital_adequacy,
        'liquidity_ratio':  liquidity_ratio,
        'roa':              roa,
        'credit_growth':    credit_growth,
        'inflation':        inflation,
        'gdp_growth':       gdp_growth,
        'current_account':  current_account,
        'banking_crisis':   crisis,
    })


df = generate_banking_data()
FEATURE_COLS = [c for c in df.columns if c != 'banking_crisis']
TARGET_COL   = 'banking_crisis'

print(f"Dataset : {df.shape[0]} observations  x  {df.shape[1]} variables")
print(f"\nClass distribution:")
vc = df[TARGET_COL].value_counts()
print(f"  No crisis (0) : {vc[0]}  ({vc[0]/len(df)*100:.1f} %)")
print(f"  Crisis    (1) : {vc[1]}  ({vc[1]/len(df)*100:.1f} %)")
print("\nFirst 5 rows:")
print(df.head().to_string())

## ── SECTION 3: EXPLORATORY DATA ANALYSIS (provided)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Banking Sector Indicators — Distribution by Crisis Status',
             fontsize=13, fontweight='bold')

colors = {0: 'steelblue', 1: 'tomato'}
for ax, feat in zip(axes.flatten(), FEATURE_COLS):
    for label, grp in df.groupby(TARGET_COL):
        ax.hist(grp[feat], bins=25, alpha=0.60,
                color=colors[label],
                label='No Crisis' if label == 0 else 'Crisis')
    ax.set_title(feat.replace('_', ' ').title())
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig('lecture14_exercise_eda.png', bbox_inches='tight')
plt.show()
print("Saved : lecture14_exercise_eda.png")

## ── SECTION 4: PREPROCESSING (provided)

In [ ]:
X = df[FEATURE_COLS].values
y = df[TARGET_COL].values

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.20, random_state=SEED, stratify=y,
)

n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
class_weight = {0: 1.0, 1: float(n_neg) / float(n_pos)}

print(f"\nTrain : {X_train.shape[0]}  |  Test : {X_test.shape[0]}")
print(f"Crisis rate — Train : {y_train.mean():.2%}  |  Test : {y_test.mean():.2%}")
print(f"Class weights       — 0: {class_weight[0]:.2f}  |  1: {class_weight[1]:.2f}")

## ── SECTION 5: BUILD THE MODEL  ◄─ YOUR TASK (TODO 1)

Build a Sequential Keras model for BINARY CLASSIFICATION.

Suggested architecture:

Dense(64,  activation='relu')  → BatchNormalization() → Dropout(0.30)

Dense(32,  activation='relu')  → BatchNormalization() → Dropout(0.30)

Dense(16,  activation='relu')

Dense(1,   activation='sigmoid')    ← sigmoid for binary output

Compile with:

optimizer = Adam(learning_rate=1e-3)

loss      = 'binary_crossentropy'

metrics   = ['accuracy', tf.keras.metrics.AUC(name='auc')]

Hint: Use class_weight (already computed above) when calling model.fit()

to compensate for the class imbalance.

In [ ]:
# WRITE YOUR CODE BELOW
# model = Sequential([
#     ...
# ])
# model.compile(...)
# model.summary()

## ── SECTION 6: TRAIN THE MODEL  ◄─ YOUR TASK (TODO 2)

Train your model with the following settings:

- EarlyStopping(monitor='val_auc', patience=20, mode='max',

restore_best_weights=True)

- class_weight=class_weight  (pass the dict computed in Section 4)

- epochs=200, batch_size=32, validation_split=0.15

In [ ]:
# WRITE YOUR CODE BELOW
# history = model.fit(
#     X_train, y_train,
#     ...
# )

## ── SECTION 7: EVALUATE & VISUALISE  ◄─ YOUR TASK (TODO 3)

Complete the following steps:

a) Plot learning curves (train vs validation loss AND train vs validation AUC)

b) Get predicted probabilities:

y_prob = model.predict(X_test).ravel()

c) Convert to class labels with threshold 0.50:

y_pred = (y_prob >= 0.50).astype(int)

d) Print the full classification report:

print(classification_report(y_test, y_pred, target_names=['No Crisis','Crisis']))

e) Plot the confusion matrix:

cm  = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(cm, display_labels=['No Crisis', 'Crisis'])

disp.plot(cmap='Blues')

plt.title('Confusion Matrix'); plt.tight_layout(); plt.show()

f) Plot the ROC curve and print the AUC score.

Starter code is provided below — uncomment it after steps (a)–(e).

In [ ]:
# WRITE YOUR CODE BELOW

# ── Starter code for ROC curve (uncomment after completing steps a–e) ────────
# fpr, tpr, _ = roc_curve(y_test, y_prob)
# auc_score   = roc_auc_score(y_test, y_prob)
#
# fig, ax = plt.subplots(figsize=(6, 5))
# ax.plot(fpr, tpr, color='tomato', lw=2, label=f'ROC curve (AUC = {auc_score:.3f})')
# ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='Random classifier')
# ax.set_xlabel('False Positive Rate')
# ax.set_ylabel('True Positive Rate')
# ax.set_title('ROC Curve — Banking Crisis Classifier')
# ax.legend()
# ax.grid(True, alpha=0.3)
# plt.tight_layout()
# plt.savefig('lecture14_exercise_roc.png', bbox_inches='tight')
# plt.show()
# print(f"AUC Score : {auc_score:.4f}")

## ── SECTION 8: DISCUSSION QUESTIONS

In [ ]:
print("""
── Discussion Questions ───────────────────────────────────────────────────────
 1. The crisis class is about 15 % of the data. How did class_weight affect
    model training? What happens if you remove it?

 2. Is accuracy or AUC a more informative metric for banking crisis prediction?
    Why might a central bank prefer one over the other?

 3. What probability threshold (0.3, 0.5, 0.7) would you recommend for a
    macro-prudential regulator?  What are the trade-offs of each threshold?

 4. Which feature has the strongest association with crisis in the EDA plots?
    Does this make economic sense?
──────────────────────────────────────────────────────────────────────────────
""")

print("=== EXERCISE COMPLETE ===")